# Assignment 2 — YOLO Object Detection on Skin-Disease Images

**Course:** Image and Video Analysis with Deep Learning (SS 2026)

**Pipeline:** load Roboflow dataset → run pre-trained **YOLO11n** baseline → fine-tune on the custom set → evaluate & compare.

All heavy lifting lives in `src/`; this notebook is the narrative walkthrough.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import torch
print('PyTorch:', torch.__version__)
print('MPS available:', torch.backends.mps.is_available())
print('CUDA available:', torch.cuda.is_available())

## Step 1 — Download the dataset

Roboflow Universe project: `skin-diseases-iyitj/skin-disease-d6tg8`. Requires `ROBOFLOW_API_KEY` in `.env`.

In [ ]:
!python ../src/download_dataset.py

## Step 2 — Model choice

We pick **YOLO11n** (nano, COCO-pretrained). Rationale:

- smallest YOLO11 variant → trains on Apple M3 MPS within the time budget;
- newer than YOLOv8n with a small mAP bump at the same speed;
- fits in unified RAM at batch=16, 640 px.

## Step 3 — Baseline inference (pre-trained on COCO)

In [ ]:
!python ../src/infer_baseline.py

In [ ]:
from IPython.display import Image
Image(str(ROOT / 'outputs' / 'comparison' / 'baseline_grid.png'))

**Observation.** COCO classes (person, dog, bottle, …) do not overlap with skin-lesion categories, so most predictions are empty or wrong-class. This is the expected baseline behavior and motivates fine-tuning.

## Step 4 — Fine-tune (transfer learning)

Train all layers (`freeze=0`) because the target domain is far from COCO. 50 epochs, 640 px, batch=16, device=MPS, seed=42, early stop after 15 stagnant epochs.

In [ ]:
!python ../src/train_finetune.py --epochs 50 --imgsz 640 --batch 16

In [ ]:
Image(str(ROOT / 'runs' / 'skin_yolo11n' / 'results.png'))

## Step 5 — Evaluate & compare

In [ ]:
!python ../src/evaluate.py

In [ ]:
import pandas as pd
df = pd.read_csv(ROOT / 'outputs' / 'metrics_comparison.csv')
df

In [ ]:
Image(str(ROOT / 'outputs' / 'comparison' / 'metrics_comparison.png'))

In [ ]:
Image(str(ROOT / 'outputs' / 'comparison' / 'side_by_side.png'))

## Discussion

See `report/report.pdf` for the full write-up: dataset details, qualitative analysis, quantitative comparison, and the Gen-AI reflection.